# Large-Scale Web Log Analysis Project

## Part 3: Gold Layer — Funnel Analysis, Anomaly Detection, Business Metrics

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

spark = SparkSession.builder \
    .appName('LogAnalysis-Part3-Gold') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

LAKE = '/home/jovyan/data/log_analysis/lake'

events = spark.read.parquet(f'{LAKE}/silver/events')
sessions = spark.read.parquet(f'{LAKE}/silver/sessions')

print(f'Silver events: {events.count():,} rows')
print(f'Silver sessions: {sessions.count():,} rows')
print(f'✅ Spark UI: http://localhost:4040')

---
## 1. Funnel Analysis

In [ ]:
# Full-period funnel
total_sessions = sessions.count()

funnel = sessions.agg(
    F.count('*').alias('all_sessions'),
    F.sum('has_product_view').alias('product_view'),
    F.sum('has_add_to_cart').alias('add_to_cart'),
    F.sum('has_checkout').alias('checkout'),
    F.sum('has_purchase').alias('purchase'),
).collect()[0]

steps = [
    ('All Sessions', funnel['all_sessions']),
    ('Product View', funnel['product_view']),
    ('Add to Cart', funnel['add_to_cart']),
    ('Checkout', funnel['checkout']),
    ('Purchase', funnel['purchase']),
]

print('=== Funnel Analysis ===')
print(f'{"Step":<14} {"Sessions":>10} {"Conv.Rate":>10} {"Drop Rate":>10}  Visual')
print('-' * 65)
prev = steps[0][1]
for i, (name, count) in enumerate(steps):
    rate = count / steps[0][1] * 100
    drop = (1 - count / prev) * 100 if i > 0 else 0
    bar = '█' * int(rate / 2)
    print(f'{name:<14} {count:>10,} {rate:>9.1f}% {drop:>9.1f}%  {bar}')
    prev = max(count, 1)

In [ ]:
# Daily funnel trend
daily_funnel = sessions.groupBy('session_date').agg(
    F.count('*').alias('sessions'),
    F.sum('has_product_view').alias('product_views'),
    F.sum('has_purchase').alias('purchases'),
).withColumn(
    'view_rate', F.round(F.col('product_views') / F.col('sessions') * 100, 1)
).withColumn(
    'purchase_rate', F.round(F.col('purchases') / F.col('sessions') * 100, 2)
).orderBy('session_date')

print('=== Daily Funnel Conversion Rates ===')
daily_funnel.show(truncate=False)

daily_funnel.write.mode('overwrite').parquet(f'{LAKE}/gold/daily_funnel')

In [ ]:
# Conversion rate comparison by device
device_funnel = sessions.groupBy('device').agg(
    F.count('*').alias('sessions'),
    F.sum('has_product_view').alias('views'),
    F.sum('has_add_to_cart').alias('carts'),
    F.sum('has_purchase').alias('purchases'),
    F.avg('session_duration_sec').alias('avg_duration'),
    F.avg('event_count').alias('avg_events'),
).withColumn(
    'view_rate', F.round(F.col('views') / F.col('sessions') * 100, 1)
).withColumn(
    'cart_rate', F.round(F.col('carts') / F.col('sessions') * 100, 1)
).withColumn(
    'purchase_rate', F.round(F.col('purchases') / F.col('sessions') * 100, 2)
).orderBy(F.col('sessions').desc())

print('=== Conversion Rates by Device ===')
device_funnel.show(truncate=False)

device_funnel.write.mode('overwrite').parquet(f'{LAKE}/gold/device_funnel')

---
## 2. Hourly and Day-of-Week Traffic Pattern Analysis

In [ ]:
# Traffic by hour
hourly = events.groupBy('event_hour').agg(
    F.count('*').alias('events'),
    F.countDistinct('user_id').alias('unique_users'),
    F.countDistinct('session_id').alias('sessions'),
).orderBy('event_hour')

print('=== Traffic by Hour ===')
hourly_data = hourly.collect()
max_events = max(r['events'] for r in hourly_data)
for r in hourly_data:
    bar = '█' * int(r['events'] / max_events * 40)
    print(f'  {r["event_hour"]:>2}:00  {r["events"]:>8,}  users {r["unique_users"]:>6,}  {bar}')

hourly.write.mode('overwrite').parquet(f'{LAKE}/gold/hourly_traffic')

In [ ]:
# Traffic by day of week
dow_names = {1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'}

daily_traffic = events \
    .withColumn('dow', F.dayofweek('timestamp')) \
    .groupBy('event_date', 'dow').agg(
        F.count('*').alias('events'),
        F.countDistinct('user_id').alias('unique_users'),
    ).orderBy('event_date')

print('=== Traffic by Day of Week ===')
daily_data = daily_traffic.collect()
max_ev = max(r['events'] for r in daily_data)
for r in daily_data:
    dow_name = dow_names.get(r['dow'], '?')
    bar = '█' * int(r['events'] / max_ev * 30)
    print(f'  {r["event_date"]} ({dow_name})  {r["events"]:>8,}  users {r["unique_users"]:>6,}  {bar}')

daily_traffic.write.mode('overwrite').parquet(f'{LAKE}/gold/daily_traffic')

---
## 3. Anomaly Detection

In [ ]:
# Anomaly detection 1: response time outliers
response_stats = events.agg(
    F.avg('response_time_ms').alias('mean'),
    F.stddev('response_time_ms').alias('std'),
    F.expr('percentile_approx(response_time_ms, 0.95)').alias('p95'),
    F.expr('percentile_approx(response_time_ms, 0.99)').alias('p99'),
).collect()[0]

print('=== Response Time Statistics ===')
print(f'  Mean:   {response_stats["mean"]:.0f}ms')
print(f'  StdDev: {response_stats["std"]:.0f}ms')
print(f'  P95:    {response_stats["p95"]}ms')
print(f'  P99:    {response_stats["p99"]}ms')

# P95 response time by hour
hourly_p95 = events.groupBy('event_date', 'event_hour').agg(
    F.expr('percentile_approx(response_time_ms, 0.95)').alias('p95_ms'),
    F.avg('response_time_ms').alias('avg_ms'),
    F.count('*').alias('requests'),
).orderBy('event_date', 'event_hour')

# Hours where P95 > 3x mean = potential incidents
threshold = response_stats['mean'] * 3
anomalies = hourly_p95.filter(F.col('p95_ms') > threshold)

anomaly_count = anomalies.count()
print(f'\nAbnormal response hours (P95 > {threshold:.0f}ms): {anomaly_count}')
if anomaly_count > 0:
    anomalies.show(10)

In [ ]:
# Anomaly detection 2: abnormal user behavior
user_stats = sessions.groupBy('user_id').agg(
    F.count('*').alias('total_sessions'),
    F.sum('event_count').alias('total_events'),
    F.avg('session_duration_sec').alias('avg_session_sec'),
    F.sum('has_purchase').alias('purchases'),
    F.countDistinct('session_date').alias('active_days'),
)

# Z-score based outlier detection
stats = user_stats.agg(
    F.avg('total_events').alias('mean_events'),
    F.stddev('total_events').alias('std_events'),
    F.avg('total_sessions').alias('mean_sessions'),
    F.stddev('total_sessions').alias('std_sessions'),
).collect()[0]

anomalous_users = user_stats.withColumn(
    'z_events', (F.col('total_events') - stats['mean_events']) / stats['std_events']
).withColumn(
    'z_sessions', (F.col('total_sessions') - stats['mean_sessions']) / stats['std_sessions']
).filter(
    (F.abs(F.col('z_events')) > 3) | (F.abs(F.col('z_sessions')) > 3)
)

print(f'=== Anomalous Users (Z-score > 3) ===')
print(f'Total users: {user_stats.count():,}, Anomalous: {anomalous_users.count():,}')
anomalous_users.orderBy(F.col('z_events').desc()).show(10)

anomalous_users.write.mode('overwrite').parquet(f'{LAKE}/gold/anomalous_users')

---
## 4. Product Behavior Aggregation (Foundation for Recommendations)

In [ ]:
# Product-level behavior aggregation
product_stats = events \
    .filter(F.col('product_id').isNotNull()) \
    .groupBy('product_id') \
    .agg(
        F.sum(F.when(F.col('event_type') == 'product_view', 1).otherwise(0)).alias('views'),
        F.sum(F.when(F.col('event_type') == 'add_to_cart', 1).otherwise(0)).alias('carts'),
        F.sum(F.when(F.col('event_type') == 'purchase', 1).otherwise(0)).alias('purchases'),
        F.countDistinct('user_id').alias('unique_viewers'),
    ) \
    .withColumn('cart_rate', F.round(F.col('carts') / F.col('views') * 100, 1)) \
    .withColumn('purchase_rate', F.round(F.col('purchases') / F.col('views') * 100, 2)) \
    .orderBy(F.col('views').desc())

print('=== Product Conversion Rates — Top 20 ===')
product_stats.show(20)

product_stats.write.mode('overwrite').parquet(f'{LAKE}/gold/product_stats')

In [ ]:
# User-product interaction matrix (input data for recommendations)
user_product = events \
    .filter(F.col('product_id').isNotNull()) \
    .groupBy('user_id', 'product_id') \
    .agg(
        F.sum(F.when(F.col('event_type') == 'product_view', 1).otherwise(0)).alias('views'),
        F.sum(F.when(F.col('event_type') == 'add_to_cart', 2).otherwise(0)).alias('cart_score'),
        F.sum(F.when(F.col('event_type') == 'purchase', 5).otherwise(0)).alias('purchase_score'),
    ) \
    .withColumn('interaction_score', F.col('views') + F.col('cart_score') + F.col('purchase_score'))

print(f'User-product interactions: {user_product.count():,}')
print(f'Unique users:    {user_product.select("user_id").distinct().count():,}')
print(f'Unique products: {user_product.select("product_id").distinct().count():,}')

user_product.write.mode('overwrite').parquet(f'{LAKE}/gold/user_product_interactions')

user_product.orderBy(F.col('interaction_score').desc()).show(10)

---
## 5. Traffic Source Analysis by Referrer

In [ ]:
# Performance by referrer
referrer_perf = sessions.groupBy('referrer').agg(
    F.count('*').alias('sessions'),
    F.sum('has_purchase').alias('purchases'),
    F.avg('event_count').alias('avg_events'),
    F.avg('session_duration_sec').alias('avg_duration_sec'),
).withColumn(
    'purchase_rate', F.round(F.col('purchases') / F.col('sessions') * 100, 2)
).withColumn(
    'session_share', F.round(F.col('sessions') / sessions.count() * 100, 1)
).orderBy(F.col('sessions').desc())

print('=== Performance by Referrer ===')
referrer_perf.show(truncate=False)

referrer_perf.write.mode('overwrite').parquet(f'{LAKE}/gold/referrer_performance')

---
## 6. Final Data Lake Structure Overview

In [ ]:
import os

print('=== Final Data Lake Structure ===')
print()
for layer in ['bronze', 'silver', 'gold']:
    layer_path = f'{LAKE}/{layer}'
    if not os.path.exists(layer_path):
        continue
    print(f'📂 {layer.upper()}/')
    for table in sorted(os.listdir(layer_path)):
        table_path = f'{layer_path}/{table}'
        if os.path.isdir(table_path):
            try:
                df = spark.read.parquet(table_path)
                count = df.count()
                cols = len(df.columns)
                print(f'  └─ {table}: {count:,} rows, {cols} columns')
            except Exception:
                print(f'  └─ {table}: (read failed)')
    print()

In [ ]:
print('''
=== Project Completion Summary ===

📊 Pipeline:
  Raw JSON (7 days, ~5 million events)
    → Bronze (Parquet conversion, partitioning)
    → Silver (bot removal, cleansing, sessionization)
    → Gold (analytics metrics)

📈 Deliverables:
  1. Funnel analysis: overall / daily / by-device conversion rates
  2. Traffic patterns: hourly and day-of-week traffic
  3. Anomaly detection: response time outliers, user behavior anomalies
  4. Product analysis: per-product conversion rates, user-product interactions
  5. Traffic source analysis: performance by referrer

🛠️ Spark techniques applied:
  - Window functions (sessionization)
  - Broadcast Join (small reference tables)
  - AQE (automatic partition optimization)
  - Parquet + partitioning (I/O optimization)
  - Anti Join (bot filtering)
  - Derived columns after aggregation (conversion rate calculation)
''')

In [ ]:
spark.stop()
print('Project complete!')